# Gold star schema template
Real joins depend on the missing ShopDB schema. Define dimensions before facts; validate FK integrity and date-rate coverage.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

def convert_to_usd(df, amount='amount_local', currency='currency', local_rate='local_to_uah', usd_rate='usd_to_uah'):
    money=DecimalType(24,6)
    df=df.withColumn('_fx_local', F.when(F.upper(F.col(currency))=='UAH',F.lit(1)).otherwise(F.col(local_rate)).cast(money))
    df=df.withColumn('_fx_usd',F.col(usd_rate).cast(money))
    invalid=df.filter(F.col('_fx_local').isNull() | F.col('_fx_usd').isNull() | (F.col('_fx_usd')<=0))
    if invalid.limit(1).count(): raise ValueError('Missing or invalid FX rates; quarantine before publishing Gold')
    return df.withColumn('amount_usd',(F.col(amount).cast(money)*F.col('_fx_local')/F.col('_fx_usd')).cast(DecimalType(24,2))).drop('_fx_local','_fx_usd')

# Stable surrogate keys: derive from business keys (e.g. deterministic SHA2), or maintain a persistent key map.
# Do NOT use monotonically_increasing_id() for MERGE-based persistent dimensions.
# Build dim_date, dim_customer, dim_product, dim_geography, dim_carrier, dim_order_profile.
# Build fact_sales at order-item grain and fact_payments at payment/refund grain.
# For each dimension/fact, use the MERGE helper from 02_silver_merge after mapping source columns.
